# Adaptive segmentation at a fiber crossing

Each fiber begins as one long segment. Persistent unresolved contact causes local subdivision; quiet, nearly straight regions are eligible for coarsening. The topology changes on the GPU while the recipe is running.

In [ ]:
from pathlib import Path
import tangle

output = Path("output")
output.mkdir(exist_ok=True)

## Two initially coarse fibers

The diameter controls capsule contact. The minimum bend radius supplies the admissible-curvature limit; it is distinct from the natural shape represented by the rest centerline.

In [ ]:
fiber = tangle.Material(
    "flexible fiber",
    diameter=20.0e-6,
    min_bend_radius=80.0e-6,
)
crossing = tangle.FiberCollection("adaptive crossing")
crossing.add_fiber([[-0.4e-3, 0.0, 0.0], [0.4e-3, 0.0, 0.0]], fiber)
crossing.add_fiber([[0.0, -0.4e-3, 0.0], [0.0, 0.4e-3, 0.0]], fiber)
print(f"{len(crossing)} fibers, initially {len(crossing)} segments")

In [ ]:
recipe = tangle.Recipe(tangle.Cell([1.0e-3] * 3))
recipe.insert(crossing, translation=[0.5e-3] * 3)
recipe.relax_until_converged(max_iterations=10_000)

## Configure adaptive refinement and coarsening

`AdaptiveSegmentationSettings()` contains the native balanced defaults. This small demonstration intentionally checks every iteration and refines after one persistent contact observation. Refinement cadence is independent of the GRASS batch size.

In [ ]:
adaptation = tangle.AdaptiveSegmentationSettings(
    refinement_interval=1,
    refinement_persistence=1,
)
settings = tangle.RelaxationSettings(
    max_iterations=12_000,
    max_step=2.0e-6,
    penetration_tolerance=0.1e-6,
    correction_fraction=0.2,
    adaptive_segmentation=adaptation,
)
adaptation.to_dict()

In [ ]:
result = recipe.run(settings)
print(result)
print(
    f"adaptive topology: {result.active_segments} active segments, "
    f"{result.segment_splits} splits, {result.segment_merges} merges"
)

## Inspect the final refinement pattern in OVITO

Coloring by refinement level makes the locally subdivided region visible.

In [ ]:
result.write_ovito(
    output / "adaptive_crossing.dump",
    view_script_path=output / "adaptive_crossing_view.py",
    session_path=output / "adaptive_crossing.ovito",
    coloring="refinement_level",
)